In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

Normalization:
Normalize a TITO object's window notation in to our defined canonical form. 

In [2]:
from __future__ import annotations
from TITO_Explore.types import TranslationInvariantTotalOrder
from TITO_Explore.normalization import process_tito_blocks

tito1 = TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[0, 4], [2]], waxing_waning=[0, 0])
tito2 = TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[4, 3], [5]], waxing_waning=[0, 0])
tito3 = TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[5, 1], [9]], waxing_waning=[1, 1]) 
tito4 = TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[1, 2], [12]], waxing_waning=[1, 1])
tito5 = TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[0, 4], [2]], waxing_waning=[1, 1])
tito6 = TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[4, 5, 6]], waxing_waning=[0, 0])
# Normalize the TITOs
print(tito1)
normalized_tito1 = process_tito_blocks(tito1)
print(normalized_tito1)

print(tito2)
normalized_tito2 = process_tito_blocks(tito2)
print(normalized_tito2)

print(tito3)
normalized_tito3 = process_tito_blocks(tito3)
print(normalized_tito3)

print(tito4)
normalized_tito4 = process_tito_blocks(tito4)
print(normalized_tito4)

print(tito5)
normalized_tito5 = process_tito_blocks(tito5)
print(normalized_tito5)

print(tito6)
normalized_tito6 = process_tito_blocks(tito6)
print(normalized_tito6)

TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[0, 4], [2]], waxing_waning=[0, 0])
[[0, 4], [2]]
TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[4, 3], [5]], waxing_waning=[0, 0])
[[0, 4], [2]]
TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[5, 1], [9]], waxing_waning=[1, 1])
[[1, 2], [0]]
TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[1, 2], [12]], waxing_waning=[1, 1])
[[1, 2], [0]]
TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[0, 4], [2]], waxing_waning=[1, 1])
[[0, 4], [2]]
TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[4, 5, 6]], waxing_waning=[0, 0])
[[0, 1, 2]]


Get Inversion Sets: 

In [3]:
from TITO_Explore.inversion_set import (
    tito_to_inversion_set,
    print_inversion_matrix,
    print_inversion_set,
)
from TITO_Explore.types import TranslationInvariantTotalOrder

tito = TranslationInvariantTotalOrder(
    n=4,
    num_blocks=2,
    vectors=[[0, 5, 6], [3]],
    waxing_waning=[0, 1],
)

print("TITO:", tito)

inversion_matrix = tito_to_inversion_set(tito)

print("\nInversion matrix:")
print_inversion_matrix(inversion_matrix)

print("\nInversion set:")
print_inversion_set(inversion_matrix)

TITO: TranslationInvariantTotalOrder(n=4, num_blocks=2, vectors=[[0, 5, 6], [3]], waxing_waning=[0, 1])
process result (0,5)
process result (0,6)
process result (1,2)

Inversion matrix:
         .        [1]        [2]          .
         .          .          .          .
         .          .          .          .
      [1]*       [2]*       [3]*       [4]*

Inversion set:
{ (0,1), (0,2), (3,4)*, (3,5)*, (3,6)*, (3,7)* }


Note- Output format: 
The output of this algorithm is a reflection table telling us what are the reflection indices in the join TITO. Here, each column represents congruence classes 0, 1, 2, 3. Same for each row. For entry (0,1), it represents the reflection index (0,1) with no star. Same applies for other entries in the table. 

Join: Find the lowest upper bound for two TITOs, whose inversion sets (in reflection table) are known based on our last algorithm demonstrated. We also developed an algorithm to convert the reflection table back to the window notation.

In [4]:
# Join test case 1: graph-based join for two non-comparable TITOs.
from TITO_Explore.inversion_set import (
    print_inversion_matrix,
    tito_to_inversion_set,
)
from TITO_Explore.join import compute_join_reflection_table, compute_tito_join
from TITO_Explore.types import TranslationInvariantTotalOrder

tito1 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[1, 0]],
    waxing_waning=[0],
)
tito2 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[2, 1]],
    waxing_waning=[0],
)

# Step 1: Convert each TITO to its inversion matrix.
print("Step 1: Inversion matrices")
inv1 = tito_to_inversion_set(tito1)
inv2 = tito_to_inversion_set(tito2)
print("\nTITO1 inversion matrix:")
print_inversion_matrix(inv1)
print("\nTITO2 inversion matrix:")
print_inversion_matrix(inv2)

# Step 2: Combine the inversion data into the join reflection table.
table_rows = compute_join_reflection_table(tito1, tito2)
print("\nStep 2: Reflection table rows (join output):")
print(f"{'Node':<10} | {'Value':<10} | {'Indicator':<10}")
print("-" * 36)
for start, value, indicator in table_rows:
    print(f"{start:<10} | {value:<10} | {indicator:<10}")

# Step 3: Convert the join reflection table back to window notation.
join_result = compute_tito_join(tito1, tito2)

if isinstance(join_result, TranslationInvariantTotalOrder):
    joined_tito = join_result
    print("\nStep 3: Comparable TITOs - larger one returned directly")
    print("  Returned TITO:", joined_tito)
else:
    blocks, waxing_waning, report = join_result
    joined_tito = TranslationInvariantTotalOrder(
        n=tito1.n,
        num_blocks=len(blocks),
        vectors=blocks,
        waxing_waning=waxing_waning,
    )
    print("\nStep 3: Reconstructed join TITO")
    print("  Blocks:", blocks)
    print("  Waxing/Waning:", waxing_waning)
    print("  Report:", report)
    print("  Reconstructed TITO:", joined_tito)

Step 1: Inversion matrices
process result (0,3)
process result (0,-1)

TITO1 inversion matrix:
         .        [1]
         .          .

TITO2 inversion matrix:
         .          .
       [1]          .
process result (0,3)
process result (0,-1)

Step 2: Reflection table rows (join output):
Node       | Value      | Indicator 
------------------------------------
0          | 2          | 1         
0          | 1          | 1         
1          | 2          | 1         
1          | 3          | 1         
process result (0,3)
process result (0,-1)

Step 3: Reconstructed join TITO
  Blocks: [[0, -1]]
  Waxing/Waning: [1]
  Report: None
  Reconstructed TITO: TranslationInvariantTotalOrder(n=2, num_blocks=1, vectors=[[0, -1]], waxing_waning=[1])


In [5]:
# When two TITOs are the same, comparable, then the larger 
# one will be returned automatically and the graph-based algorithm will not be run. 
from TITO_Explore.inversion_set import (
    print_inversion_matrix,
    tito_to_inversion_set,
)
from TITO_Explore.join import compute_join_reflection_table, compute_tito_join
from TITO_Explore.types import TranslationInvariantTotalOrder

tito1 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[0, 1]],
    waxing_waning=[0],
)
tito2 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[1, 2]],
    waxing_waning=[0],
)

print("Step 1: Inversion matrices")
inv1 = tito_to_inversion_set(tito1)
inv2 = tito_to_inversion_set(tito2)
print("\nTITO1 inversion matrix:")
print_inversion_matrix(inv1)
print("\nTITO2 inversion matrix:")
print_inversion_matrix(inv2)

table_rows = compute_join_reflection_table(tito1, tito2)
print("\nStep 2: Reflection table rows (join output):")
print(f"{'Node':<10} | {'Value':<10} | {'Indicator':<10}")
print("-" * 36)
for start, value, indicator in table_rows:
    print(f"{start:<10} | {value:<10} | {indicator:<10}")

join_result = compute_tito_join(tito1, tito2)

if isinstance(join_result, TranslationInvariantTotalOrder):
    joined_tito = join_result
    print("\nStep 3: Comparable TITOs - larger one returned directly")
    print("  Returned TITO:", joined_tito)
else:
    blocks, waxing_waning, report = join_result
    joined_tito = TranslationInvariantTotalOrder(
        n=tito1.n,
        num_blocks=len(blocks),
        vectors=blocks,
        waxing_waning=waxing_waning,
    )
    print("\nStep 3: Reconstructed join TITO")
    print("  Blocks:", blocks)
    print("  Waxing/Waning:", waxing_waning)
    print("  Report:", report)
    print("  Reconstructed TITO:", joined_tito)

Step 1: Inversion matrices
process result (0,1)
process result (0,1)

TITO1 inversion matrix:
         .          .
         .          .

TITO2 inversion matrix:
         .          .
         .          .
process result (0,1)
process result (0,1)

Step 2: Reflection table rows (join output):
Node       | Value      | Indicator 
------------------------------------
process result (0,1)
process result (0,1)

Step 3: Comparable TITOs - larger one returned directly
  Returned TITO: TranslationInvariantTotalOrder(n=2, num_blocks=1, vectors=[[1, 2]], waxing_waning=[0])


# Waning reconstruction tests

These cells test waning value recovery, waning insertion order, and a graph-based join whose reflection table is converted back using `TITO_Explore.reflection`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from TITO_Explore.inversion_set import print_inversion_matrix, print_inversion_set, tito_to_inversion_set
from TITO_Explore.join import compute_tito_join
from TITO_Explore.reflection import reflection_rows_to_blocks_and_order
from TITO_Explore.normalization import process_tito_blocks
from TITO_Explore.types import TranslationInvariantTotalOrder

In [2]:
def reflection_rows_from_inversion_matrix(inversion_matrix):
    rows = []

    for start in range(len(inversion_matrix)):
        for cell in inversion_matrix[start]:
            if cell is None:
                continue

            offsets, is_starred = cell
            for offset in offsets:
                rows.append((start, start + offset, 1 if is_starred else 0))

    return sorted(rows)


def reconstruct_single_waning_block_from_tito(tito):
    inversion_matrix = tito_to_inversion_set(tito)
    reflection_rows = reflection_rows_from_inversion_matrix(inversion_matrix)
    reconstructed_blocks, waxing_waning, report = reflection_rows_to_blocks_and_order(reflection_rows, tito.n)

    assert waxing_waning == [1]
    assert report is None
    assert len(reconstructed_blocks) == 1
    return reconstructed_blocks[0]


def assert_reconstructs_waning_block(n, block):
    tito = TranslationInvariantTotalOrder(
        n=n,
        num_blocks=1,
        vectors=[block],
        waxing_waning=[1],
    )
    expected = process_tito_blocks(tito)[0]
    reconstructed = reconstruct_single_waning_block_from_tito(tito)
    assert reconstructed == expected
    print(f"n={n}, input={block}, normalized={expected}, reconstructed={reconstructed}")

In [3]:
# Two-residue waning cases.
assert_reconstructs_waning_block(2, [0, -1])
assert_reconstructs_waning_block(2, [0, 3])
assert_reconstructs_waning_block(2, [0, -3])
assert_reconstructs_waning_block(2, [1, 6])

process result (0,-1)
n=2, input=[0, -1], normalized=[0, -1], reconstructed=[0, -1]
process result (0,3)
n=2, input=[0, 3], normalized=[0, 3], reconstructed=[0, 3]
process result (0,-3)
n=2, input=[0, -3], normalized=[0, -3], reconstructed=[0, -3]
process result (0,-7)
n=2, input=[1, 6], normalized=[0, -7], reconstructed=[0, -7]


In [4]:
# Three-residue waning cases. These check both recovered values and insertion order.
assert_reconstructs_waning_block(3, [0, 4, -1])
assert_reconstructs_waning_block(3, [0, -1, 4])
assert_reconstructs_waning_block(3, [0, 2, 4])
assert_reconstructs_waning_block(3, [0, 4, 2])
assert_reconstructs_waning_block(3, [0, -2, -1])
assert_reconstructs_waning_block(3, [0, -1, -2])
assert_reconstructs_waning_block(3, [1, 5, 0])
assert_reconstructs_waning_block(3, [2, 3, 7])

process result (0,4)
process result (0,-1)
process result (1,-4)
n=3, input=[0, 4, -1], normalized=[0, 4, -1], reconstructed=[0, 4, -1]
process result (0,4)
process result (0,-1)
process result (1,-7)
n=3, input=[0, -1, 4], normalized=[0, -1, 4], reconstructed=[0, -1, 4]
process result (0,4)
process result (0,2)
process result (1,-4)
n=3, input=[0, 2, 4], normalized=[0, 2, 4], reconstructed=[0, 2, 4]
process result (0,4)
process result (0,2)
process result (1,-1)
n=3, input=[0, 4, 2], normalized=[0, 4, 2], reconstructed=[0, 4, 2]
process result (0,-2)
process result (0,-1)
process result (1,2)
n=3, input=[0, -2, -1], normalized=[0, -2, -1], reconstructed=[0, -2, -1]
process result (0,-2)
process result (0,-1)
process result (1,-1)
n=3, input=[0, -1, -2], normalized=[0, -1, -2], reconstructed=[0, -1, -2]
process result (0,-2)
process result (0,2)
process result (1,5)
n=3, input=[1, 5, 0], normalized=[0, -2, 2], reconstructed=[0, -2, 2]
process result (0,4)
process result (0,-4)
process 

## Join tests using `compute_tito_join`

These cases test the public join API directly. Internally, `compute_tito_join` computes the join reflection table and reconstructs the resulting TITO.

In [5]:
# Join test case 1: compute_tito_join for two non-comparable waxing TITOs.
tito1 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[1, 0]],
    waxing_waning=[0],
)
tito2 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[2, 1]],
    waxing_waning=[0],
)

join_result = compute_tito_join(tito1, tito2)
blocks, waxing_waning, report = join_result
joined_tito = TranslationInvariantTotalOrder(
    n=tito1.n,
    num_blocks=len(blocks),
    vectors=blocks,
    waxing_waning=waxing_waning,
)

print("Join output:", join_result)
print("Reconstructed join TITO:", joined_tito)

assert blocks == [[0, -1]]
assert waxing_waning == [1]
assert report is None

process result (0,3)
process result (0,-1)
Join output: ([[0, -1]], [1], None)
Reconstructed join TITO: TranslationInvariantTotalOrder(n=2, num_blocks=1, vectors=[[0, -1]], waxing_waning=[1])


In [6]:
# Join/reconstruction test: n=2, [0, 3] and [2, 1]
tito1 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[0, 3]],
    waxing_waning=[0],
)
tito2 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[2, 1]],
    waxing_waning=[0],
)

join_result = compute_tito_join(tito1, tito2)
blocks, waxing_waning, report = join_result

joined_tito = TranslationInvariantTotalOrder(
    n=tito1.n,
    num_blocks=len(blocks),
    vectors=blocks,
    waxing_waning=waxing_waning,
)

print("Join output:", join_result)
print("Reconstructed join TITO:")
print(joined_tito)

join_inversion_matrix = tito_to_inversion_set(joined_tito)
print("\nJoin inversion matrix for [0, 3] and [2, 1]:")
print_inversion_matrix(join_inversion_matrix)
print("\nJoin inversion set for [0, 3] and [2, 1]:")
print_inversion_set(join_inversion_matrix)

print("\nExpected blocks if the join is special-all waning:", [[0, -1]])
print("Actual blocks returned by compute_tito_join:", blocks)
assert waxing_waning == [1]
assert report is None

process result (0,3)
process result (0,-1)
Join output: ([[0, -1]], [1], None)
Reconstructed join TITO:
TranslationInvariantTotalOrder(n=2, num_blocks=1, vectors=[[0, -1]], waxing_waning=[1])
process result (0,-1)

Join inversion matrix for [0, 3] and [2, 1]:
      [2]*       [1]*
      [1]*       [2]*

Join inversion set for [0, 3] and [2, 1]:
{ (0,2)*, (0,1)*, (1,2)*, (1,3)* }

Expected blocks if the join is special-all waning: [[0, -1]]
Actual blocks returned by compute_tito_join: [[0, -1]]


In [7]:
# Join/reconstruction test: n=2, [3, 0] waning and [1, 2] waning
tito1 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[3, 0]],
    waxing_waning=[1],
)
tito2 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[1, 2]],
    waxing_waning=[1],
)

join_result = compute_tito_join(tito1, tito2)
blocks, waxing_waning, report = join_result

joined_tito = TranslationInvariantTotalOrder(
    n=tito1.n,
    num_blocks=len(blocks),
    vectors=blocks,
    waxing_waning=waxing_waning,
)

print("Join output:", join_result)
print("Reconstructed join TITO:")
print(joined_tito)

join_inversion_matrix = tito_to_inversion_set(joined_tito)
print("\nJoin inversion matrix for [3, 0] waning and [1, 2] waning:")
print_inversion_matrix(join_inversion_matrix)
print("\nJoin inversion set for [3, 0] waning and [1, 2] waning:")
print_inversion_set(join_inversion_matrix)

assert blocks == [[0, -1]]
assert waxing_waning == [1]
assert report is None

process result (0,1)
process result (0,-3)
Join output: ([[0, -1]], [1], None)
Reconstructed join TITO:
TranslationInvariantTotalOrder(n=2, num_blocks=1, vectors=[[0, -1]], waxing_waning=[1])
process result (0,-1)

Join inversion matrix for [3, 0] waning and [1, 2] waning:
      [2]*       [1]*
      [1]*       [2]*

Join inversion set for [3, 0] waning and [1, 2] waning:
{ (0,2)*, (0,1)*, (1,2)*, (1,3)* }


Weak order comparison: Given two TITOs, the comparison function computes the local residue-pair comparisons and combines them to determine whether the first TITO is smaller, larger, equal, or incomparable in weak order.


In [6]:
from TITO_Explore.types import TranslationInvariantTotalOrder
from TITO_Explore.comparison import compare_titos_overall

tito1 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=1,
    vectors=[[0, 1]],
    waxing_waning=[0],
)

tito2 = TranslationInvariantTotalOrder(
    n=2,
    num_blocks=2,
    vectors=[[0], [1]],
    waxing_waning=[0, 0],
)

print(compare_titos_overall(tito1, tito2))

tito1 < tito2
